# Phase 1: Data Collection and Preprocessing
This phase involves loading the raw Ergast F1 datasets and filtering for the 2018-2024 seasons. Additionally we retreive weather data from Open-Meteo hisotorical API for each race and merge this dataset with our F1 dataset to get the final enrich dataset which we will use for EDA and hypothesis testing

In [2]:
import pandas as pd

data_dir = '../data/raw/'

# Load CSV files
circuits = pd.read_csv(data_dir + 'circuits.csv')
constructors = pd.read_csv(data_dir + 'constructors.csv')
drivers = pd.read_csv(data_dir + 'drivers.csv') 
qualifying = pd.read_csv(data_dir + 'qualifying.csv')
races = pd.read_csv(data_dir + 'races.csv')
results = pd.read_csv(data_dir + 'results.csv')

# Filter for 2018-2024 races
races = races[(races['year'] >= 2018) & (races['year'] <= 2024)].copy()

print(f"Successfully loaded data.")
print(f"Filtered down to {len(races)} races between 2018 and 2024.")

Successfully loaded data.
Filtered down to 149 races between 2018 and 2024.


## Step 2: Relational Merging
In this step, we merge the `results` table with `races`, `circuits`, `drivers`, and `constructors` to create a comprehensive dataset. We also calculate the `positions_gained` feature.

In [3]:
# Rename columns to avoid overlapping names during the merge
races = races.rename(columns={'name': 'race_name'})
circuits = circuits.rename(columns={'name': 'circuit_name'})
constructors = constructors.rename(columns={'name': 'constructor_name'})
drivers['driver_name'] = drivers['forename'] + ' ' + drivers['surname']


# here we merge each of our tables with our df
df = pd.merge(results, races[['raceId', 'year', 'round', 'circuitId', 'race_name', 'date']], on='raceId', how='inner')


df = pd.merge(df, circuits[['circuitId', 'circuit_name', 'lat', 'lng']], on='circuitId', how='left')

df = pd.merge(df, drivers[['driverId', 'driver_name']], on='driverId', how='left')
df = pd.merge(df, constructors[['constructorId', 'constructor_name']], on='constructorId', how='left')

df = pd.merge(df, qualifying[['raceId', 'driverId', 'position']], on=['raceId', 'driverId'], how='left', suffixes=('', '_quali'))

# Calculate Positions Gained/Lost
# 'grid' is the starting position, 'positionOrder' is the final official finishing position
df['positions_gained'] = df['grid'] - df['positionOrder']


print(f"Master Merge Complete. Total rows: {len(df)}")
display(df[['date', 'race_name', 'driver_name', 'constructor_name', 'grid', 'positionOrder', 'positions_gained']].head())

Master Merge Complete. Total rows: 2979


,date,race_name,driver_name,constructor_name,grid,positionOrder,positions_gained
0,2018-03-25,Australian Grand Prix,Sebastian Vettel,Ferrari,3,1,2
1,2018-03-25,Australian Grand Prix,Lewis Hamilton,Mercedes,1,2,-1
2,2018-03-25,Australian Grand Prix,Kimi Räikkönen,Ferrari,2,3,-1
3,2018-03-25,Australian Grand Prix,Daniel Ricciardo,Red Bull,8,4,4
4,2018-03-25,Australian Grand Prix,Fernando Alonso,McLaren,10,5,5


## Step 3: Data Enrichment
We extract unique race events from the dataset and for each item we make requests to the Open-Meteo historical API and enrich our dataset with max temptrature, mean temprature, percipiation and humidity for that event.

In [4]:
# Identify unique race events to avoid redundant API calls
unique_races = df[['raceId', 'year', 'date', 'lat', 'lng', 'circuit_name']].drop_duplicates()

print(f"Total unique race events to fetch weather for: {len(unique_races)}")

unique_races.head()

Total unique race events to fetch weather for: 149


,raceId,year,date,lat,lng,circuit_name
0,989,2018,2018-03-25,-37.8497,144.96800,Albert Park Grand Prix Circuit
20,990,2018,2018-04-08,26.0325,50.51060,Bahrain International Circuit
40,991,2018,2018-04-15,31.3389,121.22000,Shanghai International Circuit
60,992,2018,2018-04-29,40.3725,49.85330,Baku City Circuit
80,993,2018,2018-05-13,41.5700,2.26111,Circuit de Barcelona-Catalunya


In [5]:
import requests
import time

def fetch_weather_robust(lat, lng, date, race_name):
    """Fetches weather with a 10-second timeout to prevent hanging."""
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lng,
        "start_date": date,
        "end_date": date,
        "daily": ["temperature_2m_max", "temperature_2m_mean", "precipitation_sum"],
        "hourly": ["relative_humidity_2m"],
        "timezone": "auto"
    }
    try:
        response = requests.get(url, params=params, timeout=10) 
        data = response.json()
        
        max_t = data['daily']['temperature_2m_max'][0]
        mean_t = data['daily']['temperature_2m_mean'][0]
        precip = data['daily']['precipitation_sum'][0]
        
        hourly_humidity = data['hourly']['relative_humidity_2m']
        avg_h = sum(hourly_humidity) / len(hourly_humidity)
        
        return max_t, mean_t, precip, avg_h
    except Exception as e:
        print(f"Warning: Failed to fetch data for {race_name} on {date}. Skipping...")
        return None, None, None, None

# Reset index to ensure 0-149 counting
unique_races = unique_races.reset_index(drop=True)

weather_results = []
print(f"Starting robust fetch for {len(unique_races)} races...")

for index, row in unique_races.iterrows():
    # Print every race so we can track progress
    print(f"[{index+1}/149] Fetching: {row['circuit_name']} ({row['date']})...", end="\r")
    
    m_max, m_mean, rain, hum = fetch_weather_robust(row['lat'], row['lng'], row['date'], row['circuit_name'])
    
    weather_results.append({
        'raceId': row['raceId'], 
        'temp_max': m_max, 
        'temp_mean': m_mean, 
        'rainfall': rain, 
        'humidity': hum
    })
    
    time.sleep(0.1)

# Convert to df and merge
weather_df = pd.DataFrame(weather_results)
unique_races_weather = pd.merge(unique_races, weather_df, on='raceId')

print("\nEnrichment Complete.")
unique_races_weather.head()

Starting robust fetch for 149 races...
[149/149] Fetching: Yas Marina Circuit (2024-12-08)...-12-01).....
Enrichment Complete.


,raceId,year,date,lat,lng,circuit_name,temp_max,temp_mean,rainfall,humidity
0,989,2018,2018-03-25,-37.8497,144.96800,Albert Park Grand Prix Circuit,23.4,19.9,5.2,59.458333
1,990,2018,2018-04-08,26.0325,50.51060,Bahrain International Circuit,32.0,26.4,0.0,52.625000
2,991,2018,2018-04-15,31.3389,121.22000,Shanghai International Circuit,18.3,13.2,0.0,63.500000
3,992,2018,2018-04-29,40.3725,49.85330,Baku City Circuit,20.1,15.5,0.5,62.083333
4,993,2018,2018-05-13,41.5700,2.26111,Circuit de Barcelona-Catalunya,16.0,12.8,23.5,75.041667


In [6]:
# Map the weather from unique_races_weather back to every single driver result
weather_final_cols = ['raceId', 'temp_max', 'temp_mean', 'rainfall', 'humidity']
final_df = pd.merge(df, unique_races_weather[weather_final_cols], on='raceId', how='left')


print(f"Total observations before dropping: {len(final_df)}")

# Drop rows where any of our 4 weather features are missing
final_df_clean = final_df.dropna(subset=['temp_max', 'temp_mean', 'rainfall', 'humidity'])

print(f"Total observations after dropping: {len(final_df_clean)}")
print(f"Data points removed: {len(final_df) - len(final_df_clean)}")

# save final dataset
output_path = '../data/processed/f1_enriched_data.csv'
final_df_clean.to_csv(output_path, index=False)

print("Saved in processed/f1_enriched.data.csv")

Total observations before dropping: 2979
Total observations after dropping: 2839
Data points removed: 140
Saved in processed/f1_enriched.data.csv
